# Lesson 9: Proportions and A/B Tests

**Course:** hypothesis_testing_and_statistical_inference  
**Prerequisites:** Basic Python and descriptive statistics

This lesson follows a repeatable workflow: define the question, encode the design,
check assumptions, estimate the effect, quantify uncertainty, test the claim, and
communicate both statistical and practical meaning.


## Learning objectives

1. Analyze independent conversion rates with a two-proportion z-test
2. Report absolute risk difference, relative risk, odds ratio, and confidence intervals
3. Plan sample size using a minimum detectable effect
4. Recognize peeking, sample-ratio mismatch, and metric multiplicity


## Setup

Run this cell first. Every example uses a fixed random seed so results are reproducible.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Work whether the kernel starts in the course root or a level folder.
working_dir = Path.cwd()
COURSE_ROOT = working_dir if (working_dir / "data").exists() else working_dir.parent
DATA_DIR = COURSE_ROOT / "data"

rng = np.random.default_rng(20260920)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.precision", 4)


## 9.1 A/B tests are experiments, not just z-tests

The RAND Health Insurance Experiment teaching sample compares individual-deductible plans with other plans. The binary
outcome is whether a participant recorded any physician visit. This real experiment illustrates the same proportion
workflow used in online A/B testing, while the interpretation remains specific to health-insurance plans.


In [2]:
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

rand = pd.read_csv(DATA_DIR / "rand_hie_teaching_sample.csv")
summary = rand.groupby("plan_group")["any_physician_visit"].agg(["sum", "count", "mean"])
control_name, treatment_name = "Other plan", "Individual deductible"
successes = np.array([summary.loc[control_name, "sum"], summary.loc[treatment_name, "sum"]], dtype=int)
visitors = np.array([summary.loc[control_name, "count"], summary.loc[treatment_name, "count"]], dtype=int)
z, p = proportions_ztest(successes, visitors)
rates = successes/visitors
ci_control_minus_treatment = confint_proportions_2indep(
    successes[0], visitors[0], successes[1], visitors[1], method="score")
diff = rates[1]-rates[0]
diff_ci = (-ci_control_minus_treatment[1], -ci_control_minus_treatment[0])
rr = rates[1]/rates[0]
odds_ratio = (rates[1]/(1-rates[1]))/(rates[0]/(1-rates[0]))
nnt_style = 1/abs(diff) if diff else np.inf
display(summary)
pd.Series({"other-plan rate": rates[0], "deductible-plan rate": rates[1],
           "absolute difference": diff, "difference CI low": diff_ci[0],
           "difference CI high": diff_ci[1], "z": z, "p": p,
           "relative risk": rr, "odds ratio": odds_ratio,
           "people per one-event difference": nnt_style})


,sum,count,mean
plan_group,,,
Individual deductible,258,400,0.645
Other plan,268,400,0.670


other-plan rate                     0.6700
deductible-plan rate                0.6450
absolute difference                -0.0250
difference CI low                  -0.0908
difference CI high                  0.0408
z                                   0.7450
p                                   0.4563
relative risk                       0.9627
odds ratio                          0.8949
people per one-event difference    40.0000
dtype: float64

## 9.2 Design checks

Before trusting the effect:

1. Check assignment counts against the intended allocation (sample-ratio mismatch).
2. Verify one row per randomized unit and correct exposure timing.
3. Define one primary metric and a multiplicity strategy for secondary metrics.
4. Use the pre-specified duration/sample size; account for seasonality and novelty effects.
5. Report absolute lift because relative lift can exaggerate small baseline changes.


In [3]:
allocation_check = stats.chisquare(visitors, f_exp=np.repeat(visitors.sum()/2, 2))

from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
baseline = rates[0]
minimum_useful_change = .05
target = min(baseline+minimum_useful_change, .99)
h = abs(proportion_effectsize(target, baseline))
required = NormalIndPower().solve_power(h, alpha=.05, power=.80, ratio=1)
print(f"Teaching-sample allocation check p-value: {allocation_check.pvalue:.4f}")
print(f"Baseline any-visit rate: {baseline:.3f}")
print(f"For a +5 percentage-point change, required per group: {np.ceil(required):,.0f}")


Teaching-sample allocation check p-value: 1.0000
Baseline any-visit rate: 0.670
For a +5 percentage-point change, required per group: 1,329


## Worked example and interpretation

### Estimate before testing

The RAND HIE teaching subset has 400 participants in each displayed plan group. At least one physician visit occurred for 268/400 (67.0%) in the other-plan group and 258/400 (64.5%) in the individual-deductible group. The estimated **deductible minus other-plan** risk difference is −2.5 percentage points. Its score-based 95% interval extends from −9.08 to +4.08 percentage points, while the two-proportion z-test gives p = 0.456. The relative risk is 0.963 and odds ratio 0.895, but the absolute difference and its interval are easier to connect to a decision. The interval includes both a practically meaningful reduction and a small increase, so the subset does not resolve a modest plan effect.

### Check the design claim

The displayed allocation counts are exactly 400 and 400, producing an allocation chi-square p-value of 1.0. That equality is a feature of this **balanced teaching subset**, not proof that the original randomization, exposure, or outcome logging worked perfectly. A separate design calculation uses the observed 67% baseline and a prechosen +5 percentage-point change, estimating about 1,329 participants per group for 80% power under its assumptions. The current subset is much smaller. A responsible A/B readout also checks eligibility, missingness, stopping rules, and whether the metric was specified before looking at results.

## Practice and solution

**Practice.** Treatment conversion is 5.5% versus 5.0% in control, with p=.04. What else is needed
before launch?

<details><summary>Solution</summary>

Report the 0.5 percentage-point absolute lift and its CI, relative effect, costs/benefits, guardrail
metrics, assignment and data-quality checks, stopping compliance, and whether multiple metrics or
segments were searched. A small p-value alone does not establish business value or validity.
</details>

## Key takeaways

- The experimental protocol makes the causal claim credible; the z-test summarizes uncertainty.
- Absolute effects and intervals are central to decisions.
- Plan sample size from a minimum useful effect and resist unplanned peeking.


## Reproducibility checklist

- State the population, outcome, groups, and sampling unit.
- State $H_0$, $H_1$, alpha, and whether the test is one- or two-sided before looking at results.
- Verify independence from the design; use plots and diagnostics for distributional assumptions.
- Report the estimate, confidence interval, effect size, test statistic, degrees of freedom, and exact p-value.
- Separate statistical evidence from practical importance and acknowledge design limitations.
